In [1]:
import torch
import triton
import triton.language as tl
import math

assert torch.cuda.is_available(), "CUDA GPU is not available!"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)


GPU: Tesla T4
PyTorch: 2.14.1+cu130
Triton: 3.8.0


In [13]:
@triton.jit
def attention_reuse_kernel(
    Q,
    K,
    V,
    O,
    SCALE: tl.constexpr,
    N: tl.constexpr,
    NUM_Q_TILES: tl.constexpr,
    BLOCK: tl.constexpr,
):

    # Indices for a padded 4x4 matrix
    rows = tl.arange(0, BLOCK)
    cols = tl.arange(0, BLOCK)

    k_offsets = rows[:, None] * N + cols[None, :]
    v_offsets = rows[:, None] * N + cols[None, :]

    k = tl.load(
          K + k_offsets,
          mask=(rows[:, None] < N) & (cols[None, :] < N),
          other=0.0
      )

    v = tl.load(
          V + v_offsets,
          mask=(rows[:, None] < N) & (cols[None, :] < N),
          other=0.0
      )

    for tile_id in tl.static_range(0, NUM_Q_TILES):
        # Load current Q tile
        q_offsets = tile_id * N * N + rows[:, None] * N + cols[None, :]

        q = tl.load(
            Q + q_offsets,
            mask=(rows[:, None] < N) & (cols[None, :] < N),
            other=0.0
        )

        # Note: scores and output writing logic would go here
        scores = tl.dot(q, k) * SCALE

        max_scores = tl.max(scores, axis=1)

        exp_scores = tl.exp(
            scores - max_scores[:, None]
        )

        denominator = tl.sum(
            exp_scores,
            axis=1
        )

        probabilities = (
            exp_scores / denominator[:, None]
        )
        output = tl.sum(
            probabilities[:, :, None] * v[None, :, :],
            axis=1
        )
        output_offsets = (
            tile_id * N * N
            + rows[:, None] * N
            + cols[None, :]
        )

        tl.store(
            O + output_offsets,
            output,
            mask=(rows[:, None] < N) & (cols[None, :] < N)
        )


In [18]:
N = 3
NUM_Q_TILES = 4
BLOCK = 4

# Four random Q matrices
Q = torch.rand(
    (NUM_Q_TILES, N, N),
    device="cuda",
    dtype=torch.float32
)

# Fixed K matrix
K = torch.tensor(
    [
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
        [7.0, 8.0, 9.0]
    ],
    device="cuda",
    dtype=torch.float32
)

# Fixed V matrix
V = torch.tensor(
    [
        [0.1, 0.2, 0.3],
        [0.4, 0.5, 0.6],
        [0.7, 0.8, 0.9]
    ],
    device="cuda",
    dtype=torch.float32
)

O = torch.empty_like(Q)

SCALE = 1.0 / math.sqrt(N)



In [19]:
grid = (1,)
BLOCK = 16  # Overriding BLOCK to 16 to satisfy Triton's tl.dot constraint (K >= 8)

kernel = attention_reuse_kernel[grid](
    Q,
    K,
    V,
    O,
    SCALE,
    N,
    NUM_Q_TILES,
    BLOCK,
    num_warps=4
)

torch.cuda.synchronize()

In [20]:
print("\nINPUT MATRICES\n")

print("K:")
print(K)

print("\nV:")
print(V)

for i in range(NUM_Q_TILES):
    print(f"\nQ{i + 1}:")
    print(Q[i])

print("\nATTENTION OUTPUTS\n")

for i in range(NUM_Q_TILES):
    print(f"\nOutput O{i + 1}:")
    print(O[i])


INPUT MATRICES

K:
tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]], device='cuda:0')

V:
tensor([[0.1000, 0.2000, 0.3000],
        [0.4000, 0.5000, 0.6000],
        [0.7000, 0.8000, 0.9000]], device='cuda:0')

Q1:
tensor([[0.7885, 0.5896, 0.8626],
        [0.3122, 0.8138, 0.9623],
        [0.4039, 0.0521, 0.6266]], device='cuda:0')

Q2:
tensor([[0.9700, 0.9981, 0.5119],
        [0.3518, 0.1618, 0.5213],
        [0.7094, 0.9944, 0.9978]], device='cuda:0')

Q3:
tensor([[0.2760, 0.5733, 0.0992],
        [0.7958, 0.9078, 0.6333],
        [0.9644, 0.2553, 0.5591]], device='cuda:0')

Q4:
tensor([[0.2898, 0.3838, 0.8266],
        [0.6004, 0.9224, 0.7478],
        [0.8416, 0.4743, 0.7559]], device='cuda:0')

ATTENTION OUTPUTS


Output O1:
tensor([[0.6034, 0.7030, 0.8027],
        [0.5953, 0.6951, 0.7948],
        [0.4645, 0.5543, 0.6440]], device='cuda:0')

Output O2:
tensor([[0.6158, 0.7153, 0.8149],
        [0.4479, 0.5353, 0.6226],
        [0.6283, 0.7282, 0.8281]], devic

In [21]:
reference_outputs = []

for i in range(NUM_Q_TILES):

    scores = torch.matmul(Q[i], K.T) * SCALE

    probabilities = torch.softmax(scores, dim=-1)

    reference = torch.matmul(probabilities, V)

    reference_outputs.append(reference)

reference_outputs = torch.stack(reference_outputs)


In [23]:
print("\nCORRECTNESS CHECK\n")

print("Maximum absolute error:",
      (O - reference_outputs).abs().max().item())

print("Correct:",
      torch.allclose(O, reference_outputs, atol=1e-5, rtol=1e-5))


CORRECTNESS CHECK

Maximum absolute error: 0.3110424876213074
Correct: False


In [24]:
WARMUP = 100
REPEATS = 1000

# Warm-up
for _ in range(WARMUP):
    attention_reuse_kernel[grid](
        Q, K, V, O,
        SCALE, N, NUM_Q_TILES, BLOCK,
        num_warps=4
    )

torch.cuda.synchronize()

# CUDA event timing
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

In [28]:
print("\n================ K/V MEMORY TRANSFER INFORMATION ================\n")

# Logical K and V loads in one attention kernel launch
K_LOADS = 1
V_LOADS = 1

# Each matrix contains 3x3 float32 elements
K_BYTES = N * N * 4
V_BYTES = N * N * 4

TOTAL_KV_BYTES = K_BYTES + V_BYTES

print(f"K logical loads: {K_LOADS}")
print(f"V logical loads: {V_LOADS}")
print(f"Total logical K/V loads: {K_LOADS + V_LOADS}")

print(f"K data size: {K_BYTES} bytes")
print(f"V data size: {V_BYTES} bytes")
print(f"Total K/V data loaded: {TOTAL_KV_BYTES} bytes")

print(f"\nQ tiles reusing K and V: {NUM_Q_TILES}")
print("K/V reloads between Q tiles: 0")

print("\n================ TIMING NOTE ================\n")

print(f"Complete attention kernel latency: {average_us:.4f} us")
print("K/V logical loading occurs once per kernel launch.")
print("K/V values are reused across all four Q tiles.")
print("Exact HBM-to-SRAM transfer latency cannot be isolated")
print("using CUDA events.")


================ K/V MEMORY TRANSFER INFORMATION ================

K logical loads: 1
V logical loads: 1
Total logical K/V loads: 2
K data size: 36 bytes
V data size: 36 bytes
Total K/V data loaded: 72 bytes

Q tiles reusing K and V: 4
K/V reloads between Q tiles: 0

================ TIMING NOTE ================

Complete attention kernel latency: 19.7465 us
K/V logical loading occurs once per kernel launch.
K/V values are reused across all four Q tiles.
Exact HBM-to-SRAM transfer latency cannot be isolated
using CUDA events.
